# Dataset of Nanopore-based mitochondrial DNA methylation profiles associated with growth and sexual dimorphism in Nile tilapia Exploration with `mlcroissant`This notebook guides you through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.### Dataset SourceThe dataset source is provided via its Croissant schema URL:`https://sen.science/doi/10.71728/senscience.axsd-29vr/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed!pip install mlcroissant --quiet

## 1. Data LoadingLoad metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pdimport pprint# Define the Croissant schema URLcroissant_url = 'https://sen.science/doi/10.71728/senscience.axsd-29vr/fair2.json'# Load the dataset metadatadataset = mlc.Dataset(croissant_url)metadata = dataset.metadataprint(f"\nDataset Name: {metadata.name}\nDescription: {metadata.description}")print("\nDataset Metadata Overview:")pprint.pprint(metadata.to_json(), compact=True)

## 2. Data OverviewReview available record sets and their fields. All entities are referenced using their `@id` values, per FAIR^2 standards.Below, we query dataset for available record sets and fields and print their `@id`s and names for selection.

In [ ]:
# List all record sets and their fields using their @id for referencingrecord_sets = dataset.record_setsprint(f"\nNumber of record sets: {len(record_sets)}")for rs in record_sets:    print(f"\nRecordSet @id: {rs['@id']}")    print(f"RecordSet name: {rs.get('name', rs['@id'])}")    fields = rs.get('field', [])    if not isinstance(fields, list): fields = [fields]    field_ids = [f['@id'] if isinstance(f, dict) and '@id' in f else f for f in fields]    print(f"Fields (@id): {field_ids}")    # Optional: print all columns and their @id    columns = rs.get('column', [])    if not isinstance(columns, list): columns = [columns]    col_ids = [c['@id'] if isinstance(c, dict) and '@id' in c else c for c in columns]    print(f"Columns (@id): {col_ids}")

## 3. Data ExtractionLoad data from specific record sets into DataFrames for analysis.We use the record set and field `@id`s discovered above to extract data. All references use `@id` values.

In [ ]:
# Collect each record set's @idrecord_set_ids = [rs['@id'] for rs in dataset.record_sets]dataframes = {}for rs_id in record_set_ids:    try:        records = list(dataset.records(record_set=rs_id))        if records:            df = pd.DataFrame(records)            dataframes[rs_id] = df            print(f"Loaded {len(df)} records for RecordSet {rs_id}")        else:            print(f"No records found for RecordSet {rs_id}.")    except Exception as e:        print(f"Failed to load records for {rs_id}: {e}")# For demonstration, pick the first record setif dataframes:    first_rs_id = list(dataframes.keys())[0]    print(f"\nColumns for {first_rs_id}: {dataframes[first_rs_id].columns.tolist()}")    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data with respect to relevant attributes.All field references use their Croissant `@id` values.

In [ ]:
# Choose a record set to processrs_id = first_rs_iddf = dataframes[rs_id]# Print available fields (columns/@id) for processingprint(f"Fields (@id) in {rs_id}:")for col in df.columns:    print(col)# Select a numeric field for analysis# For demonstration, let's try to select fields by heuristic name search, else user will need to supply a numeric @id.numeric_fields = [col for col in df.columns if 'weight' in col.lower() or 'growth' in col.lower() or 'methylation' in col.lower() or 'position' in col.lower()]numeric_field_id = numeric_fields[0] if numeric_fields else df.columns[0]print(f"\nSelected numeric field: {numeric_field_id}")# Filter records where the field is greater than a thresholdthreshold = 10if pd.api.types.is_numeric_dtype(df[numeric_field_id]):    filtered_df = df[df[numeric_field_id] > threshold]else:    # Try converting values to numeric (errors='coerce' sets non-numeric to NaN)    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')    filtered_df = df[df[numeric_field_id] > threshold]print(f"Filtered records with {numeric_field_id} > {threshold}:")display(filtered_df.head())# Normalize the numeric field in filtered recordsnorm_field_name = f"{numeric_field_id}_normalized"filtered_df[norm_field_name] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()print(f"\nNormalized '{numeric_field_id}' for filtered records:")display(filtered_df[[numeric_field_id, norm_field_name]].head())# Group data by a categorical field, e.g., 'sex', 'group', or similar (using @id)possible_group_fields = [col for col in df.columns if 'sex' in col.lower() or 'group' in col.lower() or 'category' in col.lower()]group_field = possible_group_fields[0] if possible_group_fields else Noneif group_field:    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()    print(f"\nMean {numeric_field_id} grouped by {group_field}:")    display(grouped_df.head())else:    print("No obvious group field found; add or select manually if desired.")

## 5. VisualizationVisualize data distributions and relationships between fields to explore the dataset.Below, we plot histogram and boxplot for the numeric field, and a scatter/grouped plot if grouping is feasible.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as sns# Plot histogram of numeric fieldplt.figure(figsize=(8,4))sns.histplot(filtered_df[numeric_field_id].dropna(), kde=True)plt.title(f"Distribution of {numeric_field_id} (> {threshold})")plt.xlabel(numeric_field_id)plt.ylabel("Count")plt.show()# Boxplot (if group_field exists)if group_field:    plt.figure(figsize=(8,4))    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field_id])    plt.title(f"{numeric_field_id} by {group_field}")    plt.xlabel(group_field)    plt.ylabel(numeric_field_id)    plt.show()# Scatter plot if two numeric fields existif len(numeric_fields) > 1:    plt.figure(figsize=(8,6))    sns.scatterplot(x=filtered_df[numeric_fields[0]], y=filtered_df[numeric_fields[1]], hue=filtered_df[group_field] if group_field else None)    plt.title(f"{numeric_fields[0]} vs {numeric_fields[1]}")    plt.xlabel(numeric_fields[0])    plt.ylabel(numeric_fields[1])    plt.show()

## 6. ConclusionIn this notebook, we've:- Loaded the FAIR^2 dataset metadata and explored its structure via Croissant `@id` referencing.- Extracted tabular records for each RecordSet using `mlcroissant` and loaded them into Pandas DataFrames.- Reviewed, filtered, normalized, and grouped fields using their unique `@id`s.- Visualized numeric distributions and potential groupings.This approach provides reproducible, FAIR and transparent processing for scientific datasets.For further analyses, refer to individual `@id`s within the dataset to ensure traceable and standardized referencing throughout your workflow.